# 01 Cleaning: Steam Reviews for AI-Disclosed and Control Games
**Purpose:** Clean the raw Steam review scrape, remove the pilot (development) reviews, and draw a random sample of up to 100 negative reviews per game for classification.

**Input:** `reviews_raw.csv` (raw scrape, Nov 4, 2026), `reviews_raw_replacements.csv` (2 replacement games), `games_final.csv` (30 games), `pilot.csv` (development set)

**Output:** `reviews_clean.csv`

**Summary:** *(filled in after cleaning)*

## Setup

In [1]:
#Imports
import pandas as pd
import numpy as np

## 1. Load the Data

In [2]:
#Read in the data and confirm
df = pd.read_csv("reviews_raw.csv")
games = pd.read_csv("games_final.csv")
pilot = pd.read_csv("pilot.csv")

print(f"Loaded reviews: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Loaded games: {games.shape[0]} rows")
print(f"Loaded pilot (dev set): {pilot.shape[0]} rows")

Loaded reviews: 6612 rows, 29 columns
Loaded games: 30 rows
Loaded pilot (dev set): 312 rows


## 2. Explore the Data

In [3]:
#Explore the data structure
print("\nINFO:")
df.info()

print("\nDESCRIBE:")
print(df.describe().to_string())

print("\nDUPLICATES:")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate review IDs: {df['recommendationid'].duplicated().sum()}")

print("\nMISSING VALUES:")
print(df.isnull().sum())


INFO:
<class 'pandas.DataFrame'>
RangeIndex: 6612 entries, 0 to 6611
Data columns (total 29 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   appid                          6612 non-null   int64  
 1   recommendationid               6612 non-null   int64  
 2   review                         6612 non-null   str    
 3   voted_up                       6612 non-null   bool   
 4   votes_up                       6612 non-null   int64  
 5   votes_funny                    6612 non-null   int64  
 6   weighted_vote_score            6612 non-null   float64
 7   timestamp_created              6612 non-null   int64  
 8   language                       6612 non-null   str    
 9   steam_purchase                 6612 non-null   bool   
 10  received_for_free              6612 non-null   bool   
 11  written_during_early_access    6612 non-null   bool   
 12  refunded                       6612 non-null   bool 

In [4]:
#Explore the data: content checks specific to this scrape

# Date range: should fall within Jan 1, 2024 – Dec 31, 2025
print("DATE RANGE:")
created = pd.to_datetime(df["timestamp_created"], unit="s")
print(f"Earliest: {created.min()}, Latest: {created.max()}")

# Scrape filters: should be all negative (voted_up False) and all English
print("\nSCRAPE FILTER CHECKS:")
print(df["voted_up"].value_counts(dropna=False))
print(df["language"].value_counts(dropna=False))

# Reviews per game (smallest first) and whether all 30 games returned reviews
print("\nREVIEWS PER GAME:")
per_game = df.groupby(["group", "name"]).size().sort_values()
print(per_game.to_string())
print(f"\nGames with reviews: {df['appid'].nunique()} of {len(games)}")
missing_games = set(games["appid"]) - set(df["appid"])
print(f"Games with no reviews: {missing_games if missing_games else 'none'}")

# Reviews by group (disclosed vs. control)
print("\nREVIEWS BY GROUP:")
print(df["group"].value_counts())

# Pilot reviews in the scrape (dev set; removed later so they aren't used for testing)
print("\nPILOT (DEV SET) OVERLAP:")
in_pilot = df["recommendationid"].isin(pilot["recommendationid"])
print(f"Reviews also in pilot.csv: {in_pilot.sum()}")
print(df[in_pilot].groupby("name").size().to_string())

# Review types that may need handling: empty, very short, checkbox templates, non-English text
print("\nREVIEW TEXT:")
text = df["review"].fillna("").str.strip()
words = text.str.split().str.len()
print(f"Empty reviews: {(text == '').sum()}")
print(f"Very short reviews (1-2 words): {(words.between(1, 2)).sum()}")
print(f"Checkbox-template reviews (contain ☐ or ☑): {text.str.contains('☐|☑').sum()}")
print(f"Mostly non-Latin text (under 50% Latin letters): "
      f"{(text.str.count(r'[A-Za-z]') / text.str.len().clip(lower=1) < 0.5).sum()}")

# Review length in words
print("\nWords per review:")
print(words.describe().round(1).to_string())

DATE RANGE:
Earliest: 2024-01-01 06:47:12, Latest: 2025-12-31 22:54:31

SCRAPE FILTER CHECKS:
voted_up
False    6612
Name: count, dtype: int64
language
english    6612
Name: count, dtype: int64

REVIEWS PER GAME:
group      name                               
disclosed  Night Club Simulator                    41
control    Pharmacy Simulator                      41
disclosed  Cinema Simulator 2025                   45
           Hotel Business Simulator                48
           Shawarma Legend                         56
           Esports Godfather                       56
           FACEMINER                               85
           Serum: Toxic Survival                   85
           Chernobylite 2: Exclusion Zone          87
control    Trash Goblin                           112
disclosed  Lost Rift                              119
control    Tavern Manager Simulator               124
           Ultimate Fishing Simulator 2           128
           Motel Manager Simulator    

## Observations
- 6,612 reviews from the 30 originally selected games, 29 columns. The two replacement games (step 3) were scraped separately, so they appear under "Games with no reviews" here.- The scrape filters worked! Every review is negative, tagged English, and dated Jan 1, 2024 – Dec 31, 2025.
- Disclosed games have about half as many reviews (2,250 vs. 4,362), consistent with the review gap found in prior research. Four control games hit the 500-review cap. Sampling up to 100 per game keeps large games from dominating.
- Counts can differ by one from the earlier summary run (e.g. Motel Manager Simulator: 131 vs. 132) because Steam data changes over time. All analysis uses this dated snapshot.
- 289 reviews overlap with the pilot (dev set): all of DREADZONE (100 of 187), TRIBES 3 (100 of 464), Hotel Business Simulator (48 of 48), and Night Club Simulator (41 of 41). Removing them leaves Hotel Business and Night Club with no test reviews. 
    - Decision: both are replaced with two other low-rated AI-disclosed games (step 3), so the 312 pilot reviews stay a clean dev set.
- 277 reviews (4.2%) are only 1–2 words, and 8 are checkbox templates. These are kept: they are real negative reviews, labeled other if they give no reason.
- 53 reviews (0.8%) are mostly non-Latin characters. These could be other languages tagged as English, or ASCII art; they are inspected before deciding whether to remove them.
- Reviews have a median of 48 words (mean 100, max 1,541). The long tail matters for the cost of the Claude step.

---

## Cleaning

In [5]:
df_clean = df.copy()
print("Starting rows:", len(df_clean))

Starting rows: 6612


## 3. Replace Games With No Test Reviews

In [6]:
# Replace games whose reviews are all in the pilot (dev set)
n_before = len(df_clean)
print("Step 3 (replace games) - Starting rows:", n_before)

dropped_ids = [2738870, 2943670]  # Hotel Business Simulator, Night Club Simulator
replacements = pd.read_csv("reviews_raw_replacements.csv")
expected_drop = df_clean["appid"].isin(dropped_ids).sum()

df_clean = df_clean[~df_clean["appid"].isin(dropped_ids)]
df_clean = pd.concat([df_clean, replacements], ignore_index=True)
n_after = len(df_clean)

# Assert the swap: old games gone, new reviews added, review set matches the 30 final games
print("Step 3 (replace games) - Ending rows:", n_after,
      f"(dropped {expected_drop}, added {len(replacements)})")
assert n_after == n_before - expected_drop + len(replacements)
assert not df_clean["appid"].isin(dropped_ids).any(), "Dropped games should be gone"
assert set(df_clean["appid"]) == set(games["appid"]), "Reviews should cover exactly the 30 final games"
assert set(replacements.columns) == set(df.columns), "Replacement scrape should have the same columns"
assert df_clean["recommendationid"].is_unique, "Review IDs should be unique"

# Why: All in-window reviews for Hotel Business Simulator and Night Club Simulator were in the pilot,
# which is the dev set and can't be used for testing. Replacing them with two low-rated AI-disclosed
# management sims (REAL ESTATE Simulator, The Last FARMER) keeps 5 games in each rating tier and the same genre mix.

Step 3 (replace games) - Starting rows: 6612
Step 3 (replace games) - Ending rows: 6623 (dropped 89, added 100)
